# Tuned Lens

applies to any transformer block to give an x/logits at the last layer

In [23]:
from transformer_lens import HookedTransformer
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

model: HookedTransformer = HookedTransformer.from_pretrained("pythia-14M")

# dataset: pile-10k from neel nanda
df = pd.read_parquet("hf://datasets/NeelNanda/pile-10k/data/train-00000-of-00001-4746b8785c874cc7.parquet")

# device
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)

Loaded pretrained model pythia-14M into HookedTransformer
Moving model to device:  mps


In [24]:
# freeze base model
model.eval()
for p in model.parameters():
    p.requires_grad = False

In [25]:
# tuned-lens target: one layer
layer = 2
hook_name = f"blocks.{layer}.hook_resid_post"
d_model = model.cfg.d_model
d_vocab = model.cfg.d_vocab

In [39]:
# lens = linear translator (identity init)
lens = nn.Linear(d_model, d_model).to(device)
with torch.no_grad():
    lens.weight.copy_(torch.eye(d_model))
    lens.bias.zero_()

opt = torch.optim.AdamW(lens.parameters(), lr=1e-3)

texts = df["text"][:2000]   # tiny subset for quick prototype
seq_len = 64
steps = 2000
lr = 1e-3

In [40]:
for step in range(steps):
    text = texts[step % len(texts)]
    tokens = model.to_tokens(text, prepend_bos=True)[:, :seq_len].to(device)

    if tokens.shape[1] < 2:
        continue

    with torch.no_grad():
        final_logits = model(tokens)

        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda n: n == hook_name
        )

    resid = cache[hook_name]

    lens_logits = model.unembed(
        model.ln_final(
            lens(resid)
        )
    )

    teacher = F.softmax(final_logits[:, :-1, :], dim=-1)

    loss = F.kl_div(
        F.log_softmax(lens_logits[:, :-1, :], dim=-1),
        teacher,
        reduction="batchmean"
    )

    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 25 == 0:
        print(step, float(loss))

0 364.0825500488281
25 408.4295349121094
50 219.5004425048828
75 371.65704345703125
100 184.6610870361328
125 227.571044921875
150 233.9881591796875
175 197.95166015625
200 203.59072875976562
225 154.18519592285156
250 149.33119201660156
275 197.55213928222656
300 161.210693359375
325 205.377685546875
350 102.25004577636719
375 191.0322265625
400 164.66928100585938
425 172.00099182128906
450 306.0940856933594
475 162.52597045898438
500 163.46096801757812
525 162.2375946044922
550 157.85125732421875
575 353.2624816894531
600 189.8986053466797
625 125.40669250488281
650 235.4887237548828
675 262.67291259765625
700 70.80049133300781
725 169.1375732421875
750 249.13856506347656
775 138.230712890625
800 129.3403778076172
825 132.00091552734375
850 202.97299194335938
875 123.5848388671875
900 177.3144989013672
925 163.3023681640625
950 232.73049926757812
975 167.246337890625
1000 156.45040893554688
1025 145.60400390625
1050 123.30268859863281
1075 113.8044662475586
1100 173.54847717285156
11

In [41]:
prompt = "The sky is blue and the grass is"
tokens = model.to_tokens(prompt, prepend_bos=True)
with torch.no_grad():
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: n == hook_name)
    logits = model.unembed(model.ln_final(lens(cache[hook_name])))
pred_id = logits[0, -1].argmax().item()
print("Lens next token:", model.to_string(pred_id))

Lens next token:  a
